# Mini-Project 2 — Part 2: U-Net Training
**Pascal VOC 2007 Semantic Segmentation**

## 1. Setup

In [1]:
!pip install torch torchvision torchmetrics scipy scikit-learn matplotlib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.3 MB/s eta 0:00:00


In [ ]:
import json, os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import VOCSegmentation

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))

# Load shared config
with open('/content/project_config.json') as f:
    cfg = json.load(f)

VOC_ROOT    = cfg['voc_root']
IMG_SIZE    = tuple(cfg['img_size'])
NUM_CLASSES = cfg['num_classes']
VOC_CLASSES = cfg['voc_classes']
SAVE_DIR    = '/content/drive/MyDrive/mini_proj2_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

## 2. DataLoaders

In [ ]:
transform_img = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
transform_img_aug = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
transform_mask = transforms.Compose([
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])

# Toggle USE_AUGMENTATION for ablation study
USE_AUGMENTATION = True

train_set = VOCSegmentation(VOC_ROOT, year='2007', image_set='train', download=False,
                            transform=transform_img_aug if USE_AUGMENTATION else transform_img,
                            target_transform=transform_mask)
val_set   = VOCSegmentation(VOC_ROOT, year='2007', image_set='val',   download=False,
                            transform=transform_img, target_transform=transform_mask)

train_loader = DataLoader(train_set, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_set)} | Val/Test: {len(val_set)}')

## 3. U-Net Architecture

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=21, features=[64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups   = nn.ModuleList()
        self.pool  = nn.MaxPool2d(2, 2)

        # Encoder
        ch = in_channels
        for f in features:
            self.downs.append(DoubleConv(ch, f))
            ch = f

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Decoder
        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f * 2, f, 2, 2))
            self.ups.append(DoubleConv(f * 2, f))

        self.final = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i // 2]
            if x.shape != skip.shape:
                x = nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.ups[i + 1](x)
        return self.final(x)

model = UNet(num_classes=NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'U-Net params: {total_params:,}')

## 4. Loss Function
Toggle between cross-entropy and Dice loss for ablation study.

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, num_classes, ignore_index=255, smooth=1e-6):
        super().__init__()
        self.num_classes   = num_classes
        self.ignore_index  = ignore_index
        self.smooth        = smooth

    def forward(self, logits, targets):
        targets = targets.squeeze(1)  # (B,H,W)
        mask    = targets != self.ignore_index
        targets = targets.clone()
        targets[~mask] = 0
        probs   = torch.softmax(logits, dim=1)
        one_hot = torch.zeros_like(probs).scatter_(1, targets.unsqueeze(1), 1)
        one_hot = one_hot * mask.unsqueeze(1)  # zero out ignored pixels
        dims = (0, 2, 3)
        inter   = (probs * one_hot).sum(dims)
        union   = (probs + one_hot).sum(dims)
        dice    = (2 * inter + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()

# Toggle LOSS_FN for ablation: 'ce' or 'dice'
LOSS_FN = 'ce'

if LOSS_FN == 'ce':
    criterion = nn.CrossEntropyLoss(ignore_index=255)
else:
    criterion = DiceLoss(num_classes=NUM_CLASSES)

print(f'Loss function: {LOSS_FN}')

## 5. Training Loop

In [ ]:
NUM_EPOCHS = 30
LR         = 1e-3

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

def pixel_accuracy(preds, targets, ignore=255):
    mask   = targets != ignore
    correct = (preds == targets)[mask].sum().item()
    return correct / mask.sum().item()

def iou_score(preds, targets, num_classes, ignore=255):
    ious = []
    for c in range(num_classes):
        mask  = targets != ignore
        tp    = ((preds == c) & (targets == c) & mask).sum().item()
        fp    = ((preds == c) & (targets != c) & mask).sum().item()
        fn    = ((preds != c) & (targets == c) & mask).sum().item()
        denom = tp + fp + fn
        if denom > 0:
            ious.append(tp / denom)
    return np.mean(ious) if ious else 0.0

history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_acc': []}
best_miou = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    total_loss = 0
    for imgs, masks in train_loader:
        imgs  = imgs.to(DEVICE)
        masks = masks.long().to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        if LOSS_FN == 'ce':
            loss = criterion(logits, masks.squeeze(1))
        else:
            loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_train_loss = total_loss / len(train_loader)

    # --- Validate ---
    model.eval()
    val_loss, all_preds, all_masks = 0, [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs  = imgs.to(DEVICE)
            masks = masks.long().to(DEVICE)
            logits = model(imgs)
            if LOSS_FN == 'ce':
                loss = criterion(logits, masks.squeeze(1))
            else:
                loss = criterion(logits, masks)
            val_loss += loss.item()
            preds = logits.argmax(dim=1)
            all_preds.append(preds.cpu())
            all_masks.append(masks.squeeze(1).cpu())
    avg_val_loss = val_loss / len(val_loader)
    all_preds = torch.cat(all_preds)
    all_masks = torch.cat(all_masks)
    miou = iou_score(all_preds, all_masks, NUM_CLASSES)
    acc  = pixel_accuracy(all_preds, all_masks)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_miou'].append(miou)
    history['val_acc'].append(acc)

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
          f'Train Loss: {avg_train_loss:.4f} | '
          f'Val Loss: {avg_val_loss:.4f} | '
          f'mIoU: {miou:.4f} | Acc: {acc:.4f}')

    # Save best checkpoint
    if miou > best_miou:
        best_miou = miou
        ckpt_name = f'unet_aug{USE_AUGMENTATION}_loss{LOSS_FN}_best.pth'
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'miou': miou, 'history': history},
                   os.path.join(SAVE_DIR, ckpt_name))
        print(f'  => Saved best checkpoint (mIoU={best_miou:.4f})')

## 6. Training Curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_loss'], label='Train')
axes[0].plot(epochs, history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(epochs, history['val_miou'])
axes[1].set_title('Val mIoU'); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs, history['val_acc'])
axes[2].set_title('Val Pixel Accuracy'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'unet_training_curves.png'), dpi=150)
plt.show()
print(f'Best mIoU: {best_miou:.4f}')

## 7. Ablation: Run Variants
Re-run this notebook with different settings below to compare:

In [ ]:
# Ablation table — fill in after running each variant
ablation_results = [
    {'variant': 'U-Net, CE,   Aug=True',  'best_miou': None},
    {'variant': 'U-Net, CE,   Aug=False', 'best_miou': None},
    {'variant': 'U-Net, Dice, Aug=True',  'best_miou': None},
]

for r in ablation_results:
    print(r)